Day 5 — MLP Classifier Head From Scratch
Brain Tumour Detection Project
=====================================
Topics covered:
  1.  Linear layer internals — weight matrix, bias, what forward does
  2.  Dropout — what it zeros, train vs eval mode difference
  3.  Why no ReLU on final layer — logits vs probabilities
  4.  Softmax vs CrossEntropyLoss — why NOT to apply softmax manually
  5.  MLP class — three Linear blocks + final logit layer
  6.  BrainTumourNet — CNN + MLP combined into one nn.Module
  7.  Full forward pass — shape at every operation
  8.  Parameter count — CNN vs MLP breakdown
  9.  Logit sanity check — random init should give uniform predictions
  10. Verification checklist

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os
os.makedirs("outputs", exist_ok=True)
torch.manual_seed(42)
 

In [ ]:
# CONVBLOCK + CNN — carried forward from Day 4
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1):
        super().__init__()
        padding   = (kernel_size - 1) // 2
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size,
                              stride=stride, padding=padding, bias=False)
        self.bn   = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        nn.init.kaiming_normal_(self.conv.weight,
                                mode='fan_in', nonlinearity='relu')
        nn.init.ones_(self.bn.weight)
        nn.init.zeros_(self.bn.bias)
 
    def forward(self, x):
        return self.relu(self.bn(self.conv(x)))
 
 
class BrainTumourCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.block1 = ConvBlock(1,   32)
        self.block2 = ConvBlock(32,  64)
        self.block3 = ConvBlock(64,  128)
        self.block4 = ConvBlock(128, 256)
        self.pool   = nn.MaxPool2d(2, 2)
        self.gap    = nn.AdaptiveAvgPool2d(1)
 
    def forward(self, x):
        x = self.pool(self.block1(x))
        x = self.pool(self.block2(x))
        x = self.pool(self.block3(x))
        x = self.block4(x)
        x = self.gap(x)
        return x.view(x.size(0), -1)   # (Batches, 256)


In [4]:
# 1. LINEAR LAYER INTERNALS
"""
nn.Linear(in_features, out_features)
 
Internally holds:
  weight : tensor of shape (out_features, in_features)
  bias   : tensor of shape (out_features,)
 
Forward pass:
  output = input @ weight.T + bias
  (matrix multiply input by weight transpose, then add bias)
 
Each output neuron i computes:
  output[i] = Σⱼ weight[i,j] * input[j] + bias[i]
 
This is the dot product of the i-th weight row with the input.
Every output neuron sees EVERY input value — that's "fully connected".
"""

linear = nn.Linear(256, 128)
print(f"nn.Linear(256, 128):")
print(f"  weight shape: {tuple(linear.weight.shape)}  (out × in)")
print(f"  bias shape:   {tuple(linear.bias.shape)}")
print(f"  parameters:   {linear.weight.numel() + linear.bias.numel():,}")
print(f"                = 256x128 + 128 = {256*128}+{128} = {256*128+128:,}")

# Default initialisation of Linear
print(f"\nDefault init (Kaiming uniform):")
print(f"  weight std: {linear.weight.std():.4f}")
print(f"  bias std:   {linear.bias.std():.4f}")
print(f"  (PyTorch uses Kaiming uniform by default for Linear)")

nn.Linear(256, 128):
  weight shape: (128, 256)  (out × in)
  bias shape:   (128,)
  parameters:   32,896
                = 256x128 + 128 = 32768+128 = 32,896

Default init (Kaiming uniform):
  weight std: 0.0361
  bias std:   0.0382
  (PyTorch uses Kaiming uniform by default for Linear)


In [8]:
# 2. DROPOUT
"""
Dropout(p=0.4) during TRAINING:
  Each neuron independently zeroed with probability p=0.4
  Remaining neurons scaled up by 1/(1-p) = 1/0.6 ≈ 1.667
  The scaling keeps the expected output magnitude the same
 
  Why scaling? Without it, expected sum at training = 0.6 x sum
  but at eval (no dropout) = 1.0 x sum → magnitude mismatch
  Scaling by 1/(1-p) at training fixes this
 
During EVAL (model.eval()):
  Dropout does NOTHING — all neurons active, no scaling
  This is called "inverted dropout" — the standard PyTorch approach
 
This is why model.train() and model.eval() are NOT optional.
Wrong mode → wrong predictions.
"""

dropout = nn.Dropout(p=0.4)
x_drop  = torch.ones(1, 10)   # all ones so we can see exactly what's zeroed

dropout.train()
torch.manual_seed(5)
out_train = dropout(x_drop)
zeroed    = (out_train == 0).sum().item()
print(f"Input (all 1s):           {x_drop[0].tolist()}")
print(f"Train mode output:        {out_train[0].tolist()}")
print(f"  Zeroed: {zeroed}/10 neurons")
print(f"  Surviving values scaled to {out_train[out_train > 0][0].item():.4f}")
print(f"  (= 1 / (1 - 0.4) = {1/0.6:.4f})")

dropout.eval()
out_eval = dropout(x_drop)
print(f"\nEval mode output:         {out_eval[0].tolist()}")
print(f"  All values = 1.0 — dropout disabled during evaluation")

dropout.train()
print(f"\nDropout masks vary each forward pass (training):")
torch.manual_seed(99)
for i in range(4):
    out = dropout(x_drop)
    mask = (out > 0).int().tolist()[0]
    print(f"  Pass {i+1}: {mask}  ({sum(mask)} active)")

Input (all 1s):           [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
Train mode output:        [1.6666666269302368, 0.0, 0.0, 1.6666666269302368, 1.6666666269302368, 0.0, 0.0, 1.6666666269302368, 1.6666666269302368, 1.6666666269302368]
  Zeroed: 4/10 neurons
  Surviving values scaled to 1.6667
  (= 1 / (1 - 0.4) = 1.6667)

Eval mode output:         [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
  All values = 1.0 — dropout disabled during evaluation

Dropout masks vary each forward pass (training):
  Pass 1: [1, 1, 1, 1, 1, 1, 1, 1, 0, 1]  (9 active)
  Pass 2: [1, 1, 0, 1, 1, 1, 1, 0, 0, 1]  (7 active)
  Pass 3: [0, 1, 1, 1, 0, 0, 1, 1, 1, 0]  (6 active)
  Pass 4: [0, 1, 0, 0, 0, 1, 1, 1, 1, 1]  (6 active)


In [9]:
# 3. NO ReLU ON FINAL LAYER
"""
Final layer outputs LOGITS — raw unnormalised scores.
Example: [-1.2,  3.4,  0.8, -0.5]
          glioma  meni  pit   notum
 
These can be negative. If we applied ReLU:
  ReLU([-1.2, 3.4, 0.8, -0.5]) = [0, 3.4, 0.8, 0]
  Glioma and notumour scores forced to 0 — information destroyed
  The model can never express "confidently not glioma"
 
We need the raw signed values so:
  1. argmax → predicted class  (negative logit = low confidence)
  2. CrossEntropyLoss can apply log-softmax correctly
  3. Softmax at inference gives true probabilities
"""

logits_example = torch.tensor([[-1.2, 3.4, 0.8, -0.5]])
probs          = F.softmax(logits_example, dim=1)
pred_class     = logits_example.argmax(dim=1).item()
classes        = ["glioma", "meningioma", "pituitary", "notumour"]
 
print(f"Logits:        {logits_example[0].tolist()}")
print(f"Softmax probs: {[round(p,4) for p in probs[0].tolist()]}")
print(f"Sum of probs:  {probs[0].sum().item():.6f}  ← always 1.0")
print(f"Predicted:     {classes[pred_class]}  (index {pred_class}, highest logit)")
 
# Show ReLU would destroy information
relu_logits = F.relu(logits_example)
print(f"\nAfter ReLU:    {relu_logits[0].tolist()}")
print(f"  glioma=0, notumour=0 — we lost the negative signal")
print(f"  Model can no longer express 'definitely not this class'")

Logits:        [-1.2000000476837158, 3.4000000953674316, 0.800000011920929, -0.5]
Softmax probs: [0.0091, 0.9053, 0.0672, 0.0183]
Sum of probs:  1.000000  ← always 1.0
Predicted:     meningioma  (index 1, highest logit)

After ReLU:    [0.0, 3.4000000953674316, 0.800000011920929, 0.0]
  glioma=0, notumour=0 — we lost the negative signal
  Model can no longer express 'definitely not this class'


In [ ]:
# 4. SOFTMAX vs CROSSENTROPYLOSS

"""
CrossEntropyLoss = NLLLoss(LogSoftmax(logits), targets)
 
It applies log-softmax INTERNALLY.
If you apply softmax first then pass to CrossEntropyLoss:
  CELoss receives log(softmax(softmax(x))) — double softmax
  → wrong loss values → wrong gradients → broken training
 
Three correct vs wrong patterns:
At INFERENCE only (not training), apply softmax to get probabilities:
  probs = F.softmax(model(x), dim=1)
  pred  = probs.argmax(dim=1)
"""
logits  = torch.tensor([[2.0, 1.0, 0.5, -1.0]])
targets = torch.tensor([0])   # true class = glioma
criterion = nn.CrossEntropyLoss()
 
# Correct
loss_correct = criterion(logits, targets)
 
# Wrong — double softmax
probs_wrong  = F.softmax(logits, dim=1)
loss_wrong   = criterion(probs_wrong, targets)
 
# Manual verification of what CELoss computes
log_softmax  = F.log_softmax(logits, dim=1)
loss_manual  = -log_softmax[0, targets[0]].item()

print(f"Logits:                     {logits[0].tolist()}")
print(f"True class:                 {classes[0]} (index 0)")
print(f"\nCorrect:  criterion(logits, targets)         = {loss_correct.item():.6f}")
print(f"Manual:   -log_softmax[true_class]           = {loss_manual:.6f}")
print(f"Wrong:    criterion(softmax(logits), targets) = {loss_wrong.item():.6f}")
print(f"\n  Correct and manual match: {abs(loss_correct.item() - loss_manual) < 1e-5}")
print(f"  Wrong gives different loss → broken gradients")

Logits:                     [2.0, 1.0, 0.5, -1.0]
True class:                 glioma (index 0)

Correct:  criterion(logits, targets)         = 0.495182
Manual:   -log_softmax[true_class]           = 0.495182
Wrong:    criterion(softmax(logits), targets) = 1.052048

  Correct and manual match: True
  Wrong gives different loss → broken gradients


In [13]:
# 5. MLP CLASSIFIER HEAD
class MLPHead(nn.Module):
    """
    MLP classifier that sits on top of the CNN feature vector.
 
    Input:  (B, 256)  — feature vector from BrainTumourCNN
    Output: (B, 4)    — logits for 4 classes
 
    Architecture:
      Linear(256→128) → ReLU → Dropout(0.4)
      Linear(128→64)  → ReLU → Dropout(0.3)
      Linear(64→4)    ← raw logits, NO activation
 
    Design decisions:
      - Narrowing funnel: 256→128→64→4
        Each layer compresses and selects the most relevant features
      - Dropout before each non-final layer — regularises
      - No BatchNorm in MLP: batch stats are less meaningful on
        1D feature vectors than on 2D spatial feature maps
      - No ReLU on final layer: logits need full signed range
      - Weight init: PyTorch default (Kaiming uniform) is fine for Linear
    """
    def __init__(self, in_features: int = 256, num_classes: int = 4,
                 dropout1: float = 0.4, dropout2: float = 0.3):
        super().__init__()
 
        self.block1 = nn.Sequential(
            nn.Linear(in_features, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout1),
        )
 
        self.block2 = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout2),
        )
 
        # Final layer — logits only, no activation
        self.classifier = nn.Linear(64, num_classes)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (B, 256) feature vector from CNN
        returns: (B, 4) logits
        """
        x = self.block1(x)       # (B, 256) → (B, 128)
        x = self.block2(x)       # (B, 128) → (B,  64)
        x = self.classifier(x)   # (B,  64) → (B,   4)
        return x
    
    def param_count(self) -> dict:
        b1 = sum(p.numel() for p in self.block1.parameters())
        b2 = sum(p.numel() for p in self.block2.parameters())
        cl = sum(p.numel() for p in self.classifier.parameters())
        return {"block1": b1, "block2": b2, "classifier": cl,
                "total": b1 + b2 + cl}
    
mlp = MLPHead()
print("MLPHead created.")


x_mlp = torch.randn(4, 256)
mlp.train()
with torch.no_grad():
    out_mlp = mlp(x_mlp)
print(f"\nInput shape:  {tuple(x_mlp.shape)}")
print(f"Output shape: {tuple(out_mlp.shape)}")
 
params = mlp.param_count()
print(f"\nParameter breakdown:")
print(f"  block1 Linear(256→128)+ReLU+Drop: {params['block1']:>8,}")
print(f"  block2 Linear(128→64)+ReLU+Drop:  {params['block2']:>8,}")
print(f"  classifier Linear(64→4):           {params['classifier']:>8,}")
print(f"  Total MLP parameters:              {params['total']:>8,}")

MLPHead created.

Input shape:  (4, 256)
Output shape: (4, 4)

Parameter breakdown:
  block1 Linear(256→128)+ReLU+Drop:   32,896
  block2 Linear(128→64)+ReLU+Drop:     8,256
  classifier Linear(64→4):                260
  Total MLP parameters:                41,412


In [16]:
# 6. BrainTumourNet — CNN + MLP COMBINED
class BrainTumourNet(nn.Module):
    """
    Complete brain tumour classification model.
 
    CNN extracts spatial features from raw MRI pixels.
    MLP maps those features to class logits.
 
    Input:  (B, 1, 128, 128)  — batch of grayscale MRI images
    Output: (B, 4)            — logits for 4 tumour classes
                                [glioma, meningioma, notumour, pituitary]
 
    Usage:
      model  = BrainTumourNet()
      logits = model(images)             # training
      probs  = F.softmax(model(images), dim=1)  # inference
      pred   = probs.argmax(dim=1)       # predicted class index
    """

    def __init__(self, num_classes: int = 4):
        super().__init__()
        self.cnn = BrainTumourCNN()   # feature extractor → (B, 256)
        self.mlp = MLPHead(           # classifier        → (B, 4)
            in_features=256,
            num_classes=num_classes,
            dropout1=0.4,
            dropout2=0.3,
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.cnn(x)    # (B,1,128,128) → (B,256)
        logits   = self.mlp(features)  # (B,256) → (B,4)
        return logits
    
    def get_features(self, x: torch.Tensor) -> torch.Tensor:
        """Extract CNN features without classifying — useful for visualisation"""
        with torch.no_grad():
            return self.cnn(x)
        
    def predict(self, x: torch.Tensor) -> tuple:
        """
        Returns (predicted_class_indices, probabilities).
        Call model.eval() before using this.
        """
        self.eval()
        with torch.no_grad():
            logits = self.forward(x)
            probs  = F.softmax(logits, dim=1)
            preds  = probs.argmax(dim=1)
        return preds, probs
    
model = BrainTumourNet()
print("BrainTumourNet created.")
print(f"\nSubmodules:")
print(f"  model.cnn  → BrainTumourCNN  (feature extractor)")
print(f"  model.mlp  → MLPHead         (classifier)")


model.train()
x_full = torch.randn(4, 1, 128, 128)
with torch.no_grad():
    logits_full = model(x_full)
print(f"\nFull forward pass:")
print(f"  Input:  {tuple(x_full.shape)}")
print(f"  Output: {tuple(logits_full.shape)}")

BrainTumourNet created.

Submodules:
  model.cnn  → BrainTumourCNN  (feature extractor)
  model.mlp  → MLPHead         (classifier)

Full forward pass:
  Input:  (4, 1, 128, 128)
  Output: (4, 4)


In [28]:
# 7. FULL FORWARD PASS

# Hook-based shape tracker — records shape after every layer
shapes = {}
 
def make_hook(name):
    def hook(module, input, output):
        shapes[name] = tuple(output.shape)
    return hook

hooks = []
hooks.append(model.cnn.block1.register_forward_hook(make_hook("cnn.block1")))
hooks.append(model.cnn.pool.register_forward_hook(make_hook("cnn.pool1*")))
hooks.append(model.cnn.block2.register_forward_hook(make_hook("cnn.block2")))
hooks.append(model.cnn.block3.register_forward_hook(make_hook("cnn.block3")))
hooks.append(model.cnn.block4.register_forward_hook(make_hook("cnn.block4")))
hooks.append(model.cnn.gap.register_forward_hook(make_hook("cnn.gap")))
hooks.append(model.mlp.block1.register_forward_hook(make_hook("mlp.block1")))
hooks.append(model.mlp.block2.register_forward_hook(make_hook("mlp.block2")))
hooks.append(model.mlp.classifier.register_forward_hook(make_hook("mlp.classifier")))

model.train()
with torch.no_grad():
    _ = model(x_full)
 
# Remove hooks
for h in hooks: h.remove()
print(f"\n{'Layer':<25} {'Output shape':>20}  Notes")
print(f"{'Input':<23} {str((4,1,128,128)):>20}  raw MRI batch")
layer_notes = {
    "cnn.block1":      "32 edge feature maps",
    "cnn.pool1*":      "spatial size halved",
    "cnn.block2":      "64 texture feature maps",
    "cnn.block3":      "128 shape feature maps",
    "cnn.block4":      "256 high-level features",
    "cnn.gap":         "one value per channel",
    "mlp.block1":      "128 classification neurons",
    "mlp.block2":      "64 classification neurons",
    "mlp.classifier":  "4 logits — one per class",
}
 
for name, shape in shapes.items():
    note = layer_notes.get(name, "")
    print(f"{name:<23} {str(shape):>20}  {note}")
 
print(f"\n  CNN:  {tuple(x_full.shape)} → (4, 256)")
print(f"  MLP:  (4, 256) → (4, 4)")
print(f"  Full: {tuple(x_full.shape)} → (4, 4)")


Layer                             Output shape  Notes
Input                       (4, 1, 128, 128)  raw MRI batch
cnn.block1                 (4, 32, 128, 128)  32 edge feature maps
cnn.pool1*                  (4, 128, 16, 16)  spatial size halved
cnn.block2                   (4, 64, 64, 64)  64 texture feature maps
cnn.block3                  (4, 128, 32, 32)  128 shape feature maps
cnn.block4                  (4, 256, 16, 16)  256 high-level features
cnn.gap                       (4, 256, 1, 1)  one value per channel
mlp.block1                          (4, 128)  128 classification neurons
mlp.block2                           (4, 64)  64 classification neurons
mlp.classifier                        (4, 4)  4 logits — one per class

  CNN:  (4, 1, 128, 128) → (4, 256)
  MLP:  (4, 256) → (4, 4)
  Full: (4, 1, 128, 128) → (4, 4)


In [ ]:
# 8. PARAMETER COUNT — CNN vs MLP
cnn_params = sum(p.numel() for p in model.cnn.parameters())
mlp_params = sum(p.numel() for p in model.mlp.parameters())
total      = cnn_params + mlp_params
 
print(f"\n{'Component':<35} {'Parameters':>12} {'% of total':>12}")
print("-" * 62)
 
# CNN breakdown
for name, module in model.cnn.named_children():
    p = sum(x.numel() for x in module.parameters())
    if p > 0:
        print(f"  CNN  {name:<30} {p:>12,} {p/total*100:>11.1f}%")
 
print(f"  {'CNN total':<34} {cnn_params:>12,} {cnn_params/total*100:>11.1f}%")
print()
 
# MLP breakdown
for name, module in model.mlp.named_children():
    p = sum(x.numel() for x in module.parameters())
    print(f"  MLP  {name:<30} {p:>12,} {p/total*100:>11.1f}%")
 
print(f"  {'MLP total':<34} {mlp_params:>12,} {mlp_params/total*100:>11.1f}%")
print("-" * 62)
print(f"  {'TOTAL':<34} {total:>12,} {'100.0%':>12}")

"""
Observations:
  CNN holds {cnn_params/total*100:.0f}% of params — most work done by conv layers
  MLP holds {mlp_params/total*100:.0f}% of params — lightweight classifier on top
"""


Component                             Parameters   % of total
--------------------------------------------------------------
  CNN  block1                                  352         0.1%
  CNN  block2                               18,560         4.3%
  CNN  block3                               73,984        17.2%
  CNN  block4                              295,424        68.7%
  CNN total                               388,320        90.4%

  MLP  block1                               32,896         7.7%
  MLP  block2                                8,256         1.9%
  MLP  classifier                              260         0.1%
  MLP total                                41,412         9.6%
--------------------------------------------------------------
  TOTAL                                   429,732       100.0%


In [33]:
# 9. LOGIT SANITY CHECK
"""
Before any training, a properly initialised model should:
  - Output logits near zero (small random values)
  - Convert to probabilities near 0.25 per class (uniform)
  - Produce loss near log(4) ≈ 1.386  (log of num_classes)
 
If loss starts at 1.386 → init is correct, all classes equally likely
If loss starts very high (> 3) → something is wrong with init
If loss starts at exactly 0 → model is trivially predicting correctly
  (almost impossible, means something is seriously wrong)
"""

model.eval()
torch.manual_seed(42)
x_sanity  = torch.randn(32, 1, 128, 128)
targets_s = torch.randint(0, 4, (32,))
criterion = nn.CrossEntropyLoss()
 
with torch.no_grad():
    logits_s = model(x_sanity)
    loss_s   = criterion(logits_s, targets_s)
    probs_s  = F.softmax(logits_s, dim=1)
 
print(f"Batch size: 32,  random targets")
print(f"\nLogit stats:")
print(f"  mean:  {logits_s.mean():.4f}  (should be near 0)")
print(f"  std:   {logits_s.std():.4f}   (should be small)")
print(f"  min:   {logits_s.min():.4f}")
print(f"  max:   {logits_s.max():.4f}")

print(f"\nProbability stats (after softmax):")
print(f"  mean per class: {probs_s.mean(dim=0).tolist()}")
print(f"  (should all be near 0.25 = 1/4)")
 
print(f"\nInitial loss: {loss_s.item():.4f}")
print(f"Expected:     {np.log(4):.4f}  (= log(num_classes))")
close_to_expected = abs(loss_s.item() - np.log(4)) < 0.3
print(f"Within 0.3 of expected: {close_to_expected}")
print(f"→ {'Init looks correct ✓' if close_to_expected else 'Check init — something may be wrong'}")

# Plot logit distribution
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.patch.set_facecolor('#F8F8F6')
 
axes[0].hist(logits_s.flatten().numpy(), bins=50,
             color='#378ADD', alpha=0.85, edgecolor='none')
axes[0].set_title("Logit distribution\n(random init)", fontsize=10, fontweight='bold')
axes[0].set_xlabel("Logit value")
axes[0].set_ylabel("Count")
axes[0].axvline(0, color='#D85A30', lw=2, linestyle='--', label='zero')
axes[0].legend()
 
class_names = ["glioma", "meningioma", "notumour", "pituitary"]
mean_probs  = probs_s.mean(dim=0).numpy()
colors      = ['#E74C3C', '#3498DB', '#2ECC71', '#9B59B6']
axes[1].bar(class_names, mean_probs, color=colors, alpha=0.85)
axes[1].axhline(0.25, color='gray', lw=2, linestyle='--', label='expected (0.25)')
axes[1].set_title("Mean predicted probability\nper class (random init)",
                   fontsize=10, fontweight='bold')
axes[1].set_ylabel("Mean probability")
axes[1].set_ylim(0, 0.5)
axes[1].legend()
 
# Dropout effect on logits — train vs eval
model.train()
with torch.no_grad():
    logits_train = model(x_sanity[:8])
 
model.eval()
with torch.no_grad():
    logits_eval = model(x_sanity[:8])
 
diff = (logits_train - logits_eval).abs()
axes[2].bar(range(8), diff.mean(dim=1).numpy(),
            color='#1D9E75', alpha=0.85)
axes[2].set_title("Logit difference\ntrain mode vs eval mode (dropout effect)",
                   fontsize=10, fontweight='bold')
axes[2].set_xlabel("Sample index")
axes[2].set_ylabel("|train logit - eval logit|")
axes[2].text(0.5, 0.85, "Non-zero = dropout is active\nduring training",
             transform=axes[2].transAxes, ha='center', fontsize=9,
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
 
plt.suptitle("MLP logit sanity checks", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig("outputs/mlp_sanity.png", dpi=120, bbox_inches='tight',
            facecolor=fig.get_facecolor())

Batch size: 32,  random targets

Logit stats:
  mean:  0.0739  (should be near 0)
  std:   0.0791   (should be small)
  min:   -0.0336
  max:   0.1793

Probability stats (after softmax):
  mean per class: [0.2610638439655304, 0.22519345581531525, 0.23797841370105743, 0.2757642865180969]
  (should all be near 0.25 = 1/4)

Initial loss: 1.3764
Expected:     1.3863  (= log(num_classes))
Within 0.3 of expected: True
→ Init looks correct ✓
